# T-202: fine-tune Laya on the accepted Calvino train split (Kaggle, GPU T4 x2)

Adapted from the Laya author's notebook `notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb`
in <https://github.com/NandhaKishorM/laya> (upstream commit `6ea584941d`, Apache License 2.0,
copyright its authors). This adaptation keeps the author's training loop and changes the data, the
pinning, the seeding and the publishing. It is licensed under the same Apache License 2.0 for the
adapted parts; the rest of this repository is MIT. Spec: `docs/specs/TSD-020-laya-fine-tuning.md`.

**Kaggle notebook options:** accelerator *GPU T4 x2*, internet *On*, persistence *Off*.
**Attach** the private dataset holding `items.jsonl` and `manifest.json` (from
`scripts/export_finetune_dataset.py`). **Add the secret** `HF_TOKEN`: a fine-grained token with
write access to the model repository `kevago/calvino-laya-ft` only.

## What changed from the vendor notebook

1. `laya==0.3.24` is pinned; the vendor installs `laya>=0.1.6` and `transformers>=4.48.0`.
2. The base is the multilingual checkpoint at a pinned commit, its `model.safetensors` SHA-256 is
   checked before training; the vendor downloads the whole repository, unpinned.
3. Items come from the exporter's `items.jsonl`, not from the benchmark dataset.
4. A dropped item stops the run; the vendor discards it silently.
5. torch is seeded; the vendor never seeds it, so its exploration noise differs run to run.
6. The temperature hold-out is chosen by the exporter, by message; the vendor shuffles by item.
7. The benchmark and comparison cells are deleted: they score a different dataset.
8. The push is a new commit with a tag `t202-run<N>`, the token comes from Kaggle Secrets, and the
   pushed `model.safetensors` is checked against the local file.
9. The loop settings are written to `run_record.json` with everything else the run record needs.
10. The training-only overrides (gradient checkpointing, sequence lengths) are not saved into the
    published `rl_agent_config.json`; the vendor persists them.

The vendor's `model_name` rename is kept as the vendor writes it (see the training script).

**Reporting:** the only quality numbers this notebook prints are `train fit, not evaluation`.
Whether the checkpoint is better is decided by T-201 on the frozen test split, never here.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q "laya==0.3.24" safetensors huggingface_hub
import laya
import torch
import transformers

print("laya        :", laya.__version__)
print("torch       :", torch.__version__, "| CUDA", torch.version.cuda)
print("transformers:", transformers.__version__)
assert laya.__version__ == "0.3.24", "this run is pinned to laya 0.3.24"
assert torch.cuda.device_count() == 2, "expected two GPUs (accelerator: GPU T4 x2)"

In [ ]:
# CELL: constants
# Edit RUN_NUMBER for each run (1, 2 or 3 per the run plan) and DEVIATIONS for run 2 and 3.
RUN_NUMBER = 1
LOSS_MODE = "full"  # run 3 only: "ce_only" (the run plan's soft cross-entropy run)
DEVIATIONS = []  # run 2/3: [{"setting": "epochs", "reason": "run plan: rule R"}]
EPOCHS_OVERRIDE = None  # run 2 only: 8

EXPORT_DIR = "/kaggle/input/calvino-t202-export"  # the attached private dataset

# The pinned base (classifiers.yaml `base`); a test keeps these equal to that file.
BASE_REPO = "convaiinnovations/laya"
BASE_SUBFOLDER = "multilingual"
BASE_REVISION = "7b928d828b7b0e022f929d9bd2e44165aa270148"
BASE_SHA256 = "9d628fd971b700382ac6f65920a86f149777b2e748e0c955fb3b19695aa8f204"

OUTPUT_REPO = "kevago/calvino-laya-ft"
NOTEBOOK_PATH = "notebooks/t202_laya_finetune_kaggle.ipynb"
WORK = "/kaggle/working"
OUTPUT_DIR = f"{WORK}/laya_finetuned"
RULE_R_MIN_FIT = 0.90  # run plan: carry a run into T-201 only if train fit >= 90% on both

In [ ]:
# CELL: verify-export
import hashlib
import json
import os


def verify_export(export_dir):
    """Check the attached export is the one the manifest describes. Returns (rows, manifest)."""
    with open(os.path.join(export_dir, "manifest.json"), encoding="utf-8") as handle:
        manifest = json.load(handle)
    raw = open(os.path.join(export_dir, "items.jsonl"), "rb").read()
    assert hashlib.sha256(raw).hexdigest() == manifest["items_sha256"], "items.jsonl changed"
    rows = [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]
    questions = {}
    for row in rows:
        questions.setdefault(row["question_id"], row["question"])
        assert questions[row["question_id"]] == row["question"], "question differs between items"
    canonical = json.dumps(
        {"needs_human": questions["needs_human"], "workflow_area": questions["workflow_area"]},
        ensure_ascii=False,
        separators=(",", ":"),
    )
    schema_hash = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
    assert schema_hash == manifest["question_schema_sha256"], (
        "the questions in the export are not the ones the manifest names"
    )
    assert all(check["passed"] for check in manifest["leakage_guard"]), "leakage guard did not pass"
    return rows, manifest


if __name__ == "__main__":
    EXPORT_ROWS, MANIFEST = verify_export(EXPORT_DIR)
    print(len(EXPORT_ROWS), "items;", MANIFEST["roles"], "| excluded:", MANIFEST["excluded"])

In [ ]:
# CELL: base-download
from huggingface_hub import snapshot_download

snapshot = snapshot_download(
    BASE_REPO, revision=BASE_REVISION, allow_patterns=[f"{BASE_SUBFOLDER}/*"]
)
assert os.path.basename(snapshot) == BASE_REVISION, "the snapshot is not the pinned commit"
MODEL_DIR = os.path.join(snapshot, BASE_SUBFOLDER)
for name in ("model.safetensors", "rl_agent_config.json", "encoder", "tokenizer"):
    assert os.path.exists(os.path.join(MODEL_DIR, name)), f"missing {name}"

digest = hashlib.sha256()
with open(os.path.join(MODEL_DIR, "model.safetensors"), "rb") as handle:
    for chunk in iter(lambda: handle.read(1 << 20), b""):
        digest.update(chunk)
assert digest.hexdigest() == BASE_SHA256, "base model.safetensors is not the pinned digest"
print("base verified:", BASE_REPO, BASE_SUBFOLDER, BASE_REVISION[:12])

In [ ]:
%%writefile /kaggle/working/t202_lib.py
"""Shared pieces of the T-202 run (TSD-020): items, batching, the loss, the temperature fit.

Adapted from the Laya author's fine-tuning notebook (https://github.com/NandhaKishorM/laya,
Apache License 2.0). The loss and the temperature fit are the author's; items come from the
Calvino export, and nothing here reads an external dataset.
"""
import torch
from laya.common import QTYPES, TEMP_MAX, TEMP_MIN, build_sequence, proper_reward, render_options

# Settings the author fixes in code; the run record writes them down (TSD-020 change 9).
SETTINGS = {
    "epochs": 4,
    "micro_batch": 8,
    "grad_accum": 4,
    "group_size": 4,
    "lr_encoder": 2.5e-5,
    "lr_head": 1.0e-4,
    "weight_decay": 0.01,
    "schedule": "cosine",
    "min_lr": 1e-6,
    "grad_clip": 1.0,
    "seed": 42,                 # shuffle seed base: 42 + epoch + rank
    "torch_seed_base": 42,      # torch is seeded with this + rank (the author never seeds torch)
    "cross_entropy_weight": 1.0,
    "sigma_start": 0.4,
    "sigma_end": 0.1,
    "reward_weight_spherical": 0.75,
    "reward_weight_rps": 1.0,
    "max_len": 1024,
    "head_max_len": 256,
    "max_tokens_per_batch": 4096,
    "gradient_checkpointing": True,
    "mixed_precision": "fp16",
    "holdout_max_items": 400,
    "holdout_fraction_percent": 10,
}


def build_training_item(tok, row):
    """One export row -> one training item, or None when the marker count does not match."""
    question = row["question"]
    assert question["type"] == "choice", "the first slice has choice questions only"
    criteria = question["criteria"]
    keys = list(criteria.keys())
    if row["target_option"] not in keys:
        raise ValueError(f"{row['item_id']}: target {row['target_option']!r} is not an option")
    target = [1.0 if key == row["target_option"] else 0.0 for key in keys]
    expected_markers = len(render_options({"t": "choice", "crit": criteria}))
    sequence, markers = build_sequence(
        tok,
        row["state"],
        {"t": "choice", "ins": question["instructions"], "crit": criteria},
        SETTINGS["max_len"],
        SETTINGS["head_max_len"],
    )
    if len(markers) != expected_markers:
        return None
    return {
        "ids": sequence,
        "markers": markers,
        "qtype": QTYPES["choice"],
        "target": target,
        "label": keys.index(row["target_option"]),
        "qid": row["question_id"],
        "msg_id": row["msg_id"],
        "item_id": row["item_id"],
    }


def build_items(tok, rows):
    """Split the export by its role column. Returns (train, holdout, dropped_item_ids)."""
    train, holdout, dropped = [], [], []
    for row in rows:
        item = build_training_item(tok, row)
        if item is None:
            dropped.append(row["item_id"])
        elif row["role"] == "train":
            train.append(item)
        elif row["role"] == "laya_temperature_holdout":
            holdout.append(item)
        else:
            raise ValueError(f"{row['item_id']}: unknown role {row['role']!r}")
    return train, holdout, dropped


def collate_train_batch(items, pad_id):
    n, length = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, length), pad_id, dtype=torch.long)
    att = torch.zeros((n, length), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids,
        "attention_mask": att,
        "marker_pos": mpos,
        "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items]),
    }


def compute_loss(logits, act, mask, target, qtype, sigma, loss_mode="full"):
    """The author's loss: a policy-gradient term on noisy logits plus soft cross-entropy.

    ``loss_mode="ce_only"`` drops the policy-gradient term (run 3 of the run plan).
    Returns (loss before gradient-accumulation scaling, mean reward).
    """
    group = SETTINGS["group_size"]
    k = mask.sum(-1, keepdim=True).float()
    log_probs = torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)
    loss_ce = -(target * log_probs).sum(-1).mean()
    # Keeps `act` in the graph so DDP does not see an unused output.
    keep_graph = 0.0 * act.sum()
    if loss_mode == "ce_only":
        return SETTINGS["cross_entropy_weight"] * loss_ce + keep_graph, torch.zeros(())
    eps = torch.randn((group,) + logits.shape, device=logits.device) * sigma * mask
    eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
    z = logits.detach().unsqueeze(0) + eps
    q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
    with torch.no_grad():
        reward = proper_reward(
            q,
            target.unsqueeze(0),
            qtype,
            mask,
            w_sph=SETTINGS["reward_weight_spherical"],
            w_rps=SETTINGS["reward_weight_rps"],
        )
        advantage = reward - reward.mean(0, keepdim=True)
        advantage = advantage / (advantage.std() + 1e-6)
    logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma**2)
    loss_rl = -(advantage * logp).mean()
    loss = loss_rl + SETTINGS["cross_entropy_weight"] * loss_ce + keep_graph
    return loss, reward.mean()


def fit_one_temp(selected):
    """The author's one-parameter temperature fit on (logits, target) pairs."""
    if len(selected) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in selected)
    z_all = torch.full((len(selected), kmax), -1e4)
    t_all = torch.zeros((len(selected), kmax))
    for i, (z, t) in enumerate(selected):
        z_all[i, : len(z)] = torch.tensor(z)
        t_all[i, : len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    optimizer = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)

    def closure():
        optimizer.zero_grad()
        loss = -(t_all * torch.log_softmax(z_all / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(torch.clamp(log_t.exp(), TEMP_MIN, TEMP_MAX).item())

In [ ]:
# CELL: items
import sys

from laya.agent import _fix_tokenizer_config
from transformers import AutoTokenizer

sys.path.insert(0, WORK)
import t202_lib  # noqa: E402  (written by the cell above)

_fix_tokenizer_config(MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(os.path.join(MODEL_DIR, "tokenizer"))
train_items, holdout_items, dropped = t202_lib.build_items(tokenizer, EXPORT_ROWS)
assert not dropped, f"{len(dropped)} item(s) dropped (marker count mismatch): {dropped[:5]}"
assert len(train_items) == MANIFEST["roles"]["train"]
assert len(holdout_items) == MANIFEST["roles"]["laya_temperature_holdout"]
torch.save(train_items, f"{WORK}/train_items.pt")
torch.save(holdout_items, f"{WORK}/holdout_items.pt")
print(len(train_items), "train items;", len(holdout_items), "hold-out items; none dropped")

In [ ]:
%%writefile /kaggle/working/train_ddp.py
"""DDP training loop for T-202 (TSD-020). Adapted from the Laya author's notebook (Apache 2.0).

Usage: torchrun --standalone --nproc_per_node=2 train_ddp.py MODEL_DIR OUTPUT_DIR [full|ce_only] [EPOCHS]
"""
import json
import os
import random
import sys
import time

import torch
import torch.distributed as dist
from laya.common import build_model
from safetensors.torch import load_file, save_file
from torch.nn.parallel import DistributedDataParallel as DDP
from transformers import AutoTokenizer

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import t202_lib
from t202_lib import SETTINGS


def main():
    model_dir, output_dir = sys.argv[1], sys.argv[2]
    loss_mode = sys.argv[3] if len(sys.argv) > 3 else "full"
    epochs = int(sys.argv[4]) if len(sys.argv) > 4 else SETTINGS["epochs"]
    assert loss_mode in ("full", "ce_only")

    dist.init_process_group("nccl")
    rank, world_size = dist.get_rank(), dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)
    # The author never seeds torch, so the exploration noise differs between runs (TSD-020 change 5).
    torch.manual_seed(SETTINGS["torch_seed_base"] + rank)
    torch.cuda.manual_seed_all(SETTINGS["torch_seed_base"] + rank)

    with open(os.path.join(model_dir, "rl_agent_config.json")) as handle:
        saved_config = json.load(handle)
    # Training-only overrides live on a copy; the published config keeps the saved values (change 10).
    cfg = dict(saved_config)
    cfg["gradient_checkpointing"] = SETTINGS["gradient_checkpointing"]
    cfg["max_tokens_per_batch"] = SETTINGS["max_tokens_per_batch"]
    cfg["max_len"] = SETTINGS["max_len"]
    cfg["head_max_len"] = SETTINGS["head_max_len"]

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    model.load_state_dict(load_file(os.path.join(model_dir, "model.safetensors")), strict=True)
    parameter_count = sum(p.numel() for p in model.parameters())
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()
    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)

    work = os.path.dirname(os.path.abspath(__file__))
    train_items = torch.load(os.path.join(work, "train_items.pt"), weights_only=False)
    holdout_items = torch.load(os.path.join(work, "holdout_items.pt"), weights_only=False)
    # Every rank needs the same count or DDP deadlocks on an unmatched all-reduce
    # (https://github.com/NandhaKishorM/laya/issues/678); this drops at most world_size - 1 items.
    used = train_items[: len(train_items) // world_size * world_size]
    my_items = used[rank::world_size]

    micro, accum = SETTINGS["micro_batch"], SETTINGS["grad_accum"]
    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]
    optimizer = torch.optim.AdamW(
        [
            {"params": enc_params, "lr": SETTINGS["lr_encoder"]},
            {"params": head_params, "lr": SETTINGS["lr_head"]},
        ],
        weight_decay=SETTINGS["weight_decay"],
    )
    total_updates = (len(my_items) // (micro * accum)) * epochs
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, total_updates), eta_min=SETTINGS["min_lr"]
    )
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    if rank == 0:
        print(
            f"T-202 training: {len(used)} train items ({len(holdout_items)} held out for laya's "
            f"temperatures) | {len(my_items)} per rank | {epochs} epochs | loss {loss_mode}"
        )
    started = time.time()
    epoch_losses = []
    for epoch in range(epochs):
        random.seed(SETTINGS["seed"] + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches, accum_step = 0.0, 0, 0
        optimizer.zero_grad(set_to_none=True)
        progress = epoch / max(1, epochs - 1)
        sigma = SETTINGS["sigma_start"] + (SETTINGS["sigma_end"] - SETTINGS["sigma_start"]) * progress
        for start in range(0, len(my_items), micro):
            chunk = my_items[start : start + micro]
            if not chunk:
                continue
            batch = t202_lib.collate_train_batch(chunk, tok.pad_token_id)
            mask = batch["marker_mask"].to(device)
            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    mask,
                    batch["qtype"].to(device),
                )
            loss, reward = t202_lib.compute_loss(
                logits.float(), act, mask, batch["target"].to(device), batch["qtype"].to(device),
                sigma, loss_mode,
            )
            scaler.scale(loss / accum).backward()
            accum_step += 1
            if accum_step % accum == 0 or (start + micro) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), SETTINGS["grad_clip"])
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            epoch_loss += loss.item()
            n_batches += 1
            if rank == 0 and n_batches % 50 == 0:
                print(f"  epoch {epoch + 1}/{epochs} step {n_batches} loss {loss.item():.4f} "
                      f"reward {float(reward):.3f} lr {scheduler.get_last_lr()[0]:.2e}")
        epoch_losses.append(epoch_loss / max(1, n_batches))
        if rank == 0:
            print(f"=== epoch {epoch + 1}/{epochs} done in {time.time() - started:.1f}s | "
                  f"average loss {epoch_losses[-1]:.4f} (rank 0) ===")
        dist.barrier()

    dist.barrier()
    if rank == 0:
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache()
        model.eval()

        def logits_for(items):
            out = []
            with torch.no_grad():
                for start in range(0, len(items), 16):
                    chunk = items[start : start + 16]
                    batch = t202_lib.collate_train_batch(chunk, tok.pad_token_id)
                    with torch.autocast("cuda", dtype=torch.float16):
                        sub, _ = model(
                            batch["input_ids"].to(device), batch["attention_mask"].to(device),
                            batch["marker_pos"].to(device), batch["marker_mask"].to(device),
                            batch["qtype"].to(device),
                        )
                    array = sub.float().cpu().numpy()
                    out += [array[r, : len(it["markers"])] for r, it in enumerate(chunk)]
            return out

        # laya's own temperatures, fitted on the hold-out slice the model never trained on.
        temps = {"choice": 1.2, "score": 1.2, "noul": 1.2}   # the author's fallback values
        pairs = list(zip(logits_for(holdout_items), holdout_items))
        selected = [(z, it["target"]) for z, it in pairs if it["qtype"] == 0]
        if selected:
            temps["choice"] = t202_lib.fit_one_temp(selected)
        print("laya temperatures (choice, score, noul):", {k: round(v, 3) for k, v in temps.items()})

        # Training-set fit per question: reported as "train fit, not evaluation".
        fit = {}
        for qid in sorted({it["qid"] for it in used}):
            subset = [it for it in used if it["qid"] == qid]
            predicted = [int(z.argmax()) for z in logits_for(subset)]
            fit[qid] = sum(p == it["label"] for p, it in zip(predicted, subset)) / len(subset)
        print("train fit, not evaluation:", {k: round(v, 4) for k, v in fit.items()})

        os.makedirs(output_dir, exist_ok=True)
        weights = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
        save_file(weights, os.path.join(output_dir, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(output_dir, "encoder"))
        tok.save_pretrained(os.path.join(output_dir, "tokenizer"))
        published = dict(saved_config)
        published["fine_tuned"] = True
        published["model_name"] = "laya-typed-decisions"   # as the author writes it; see the header
        published["temperature"] = [temps["choice"], temps["score"], temps["noul"]]
        published.pop("temperature_by_options", None)
        with open(os.path.join(output_dir, "rl_agent_config.json"), "w") as handle:
            json.dump(published, handle, indent=2)
        with open(os.path.join(output_dir, "training_log.json"), "w") as handle:
            json.dump(
                {
                    "loss_mode": loss_mode,
                    "epochs": epochs,
                    "loss_per_epoch": epoch_losses,
                    "wall_time_seconds": time.time() - started,
                    "train_fit": fit,
                    "laya_temperatures": temps,
                    "parameter_count": parameter_count,
                    "train_items_used": len(used),
                    "holdout_items": len(holdout_items),
                    "settings": SETTINGS,
                },
                handle,
                indent=2,
            )
        print("saved to", output_dir)
    dist.destroy_process_group()


if __name__ == "__main__":
    main()

In [ ]:
# CELL: launch
args = f"{MODEL_DIR} {OUTPUT_DIR} {LOSS_MODE}" + (f" {EPOCHS_OVERRIDE}" if EPOCHS_OVERRIDE else "")
!torchrun --standalone --nproc_per_node=2 {WORK}/train_ddp.py {args}

In [ ]:
# CELL: log
with open(f"{OUTPUT_DIR}/training_log.json", encoding="utf-8") as handle:
    LOG = json.load(handle)
print("loss per epoch (rank 0):", [round(x, 4) for x in LOG["loss_per_epoch"]])
print("train fit, not evaluation:", {k: round(v, 4) for k, v in LOG["train_fit"].items()})
print("laya temperatures (hold-out fit, not Calvino's calibration):", LOG["laya_temperatures"])
RULE_R_PASSED = bool(LOG["train_fit"]) and all(
    v >= RULE_R_MIN_FIT for v in LOG["train_fit"].values()
)
print("rule R (train fit >= 90% on both questions):", "PASS" if RULE_R_PASSED else "FAIL")
if not RULE_R_PASSED:
    print("Do not carry this run into T-201: follow the run plan (TSD-020, decision 42).")

In [ ]:
# CELL: push
import textwrap

from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
api.create_repo(OUTPUT_REPO, repo_type="model", exist_ok=True)
TAG = f"t202-run{RUN_NUMBER}"
assert TAG not in {ref.name for ref in api.list_repo_refs(OUTPUT_REPO).tags}, (
    f"{TAG} already exists: a published run is never overwritten"
)

CARD = textwrap.dedent(f"""\
    ---
    license: apache-2.0
    base_model: {BASE_REPO}
    tags: [laya, calvino, synthetic-data]
    ---
    # Calvino Laya fine-tune, run {RUN_NUMBER}

    A fine-tune of `{BASE_REPO}` (`{BASE_SUBFOLDER}`, commit `{BASE_REVISION}`) for
    Project Calvino's stuck-payments workflow, trained on **synthetic**, team-generated
    Spanish customer messages for two questions only: `needs_human` and `workflow_area`.
    Apache License 2.0.

    **Not for production decisions.** The only numbers recorded for this run are
    training-set fit ("train fit, not evaluation"); whether it is better than the base
    checkpoint is decided by a separate held-out comparison. See the Calvino repository,
    TSD-020.
    """)
with open(f"{OUTPUT_DIR}/README.md", "w", encoding="utf-8") as handle:
    handle.write(CARD)

commit = api.upload_folder(
    folder_path=OUTPUT_DIR, repo_id=OUTPUT_REPO, commit_message=f"T-202 run {RUN_NUMBER}"
)
OUTPUT_REVISION = commit.oid
api.create_tag(OUTPUT_REPO, tag=TAG, revision=OUTPUT_REVISION)

digest = hashlib.sha256()
with open(f"{OUTPUT_DIR}/model.safetensors", "rb") as handle:
    for chunk in iter(lambda: handle.read(1 << 20), b""):
        digest.update(chunk)
OUTPUT_SHA256 = digest.hexdigest()
info = api.model_info(OUTPUT_REPO, revision=OUTPUT_REVISION, files_metadata=True)
remote = next(s for s in info.siblings if s.rfilename == "model.safetensors")
assert remote.lfs.sha256 == OUTPUT_SHA256, "the pushed model.safetensors differs from the local one"
print("pushed", OUTPUT_REPO, "commit", OUTPUT_REVISION, "tag", TAG)
print("model.safetensors sha256", OUTPUT_SHA256)

In [ ]:
# CELL: assemble-run-record
import json  # noqa: F811  (this cell also runs on its own in the tests)

LOSS_TEXT = {"full": "policy gradient + soft cross-entropy", "ce_only": "soft cross-entropy only"}
DATA_FIELDS = (
    "input_sha256",
    "set_version",
    "rubric_version",
    "question_schema_sha256",
    "counts",
    "reviewed",
    "unreviewed",
    "excluded",
    "exporter_git_sha",
)


def assemble_run_record(
    *, run_number, run_date, notebook, base, output, environment, manifest, log, deviations, notes
):
    """The dict `calvino.classifiers.finetune_record.FineTuneRunRecord` validates."""
    s = log["settings"]
    return {
        "run": run_number,
        "run_date": run_date,
        "notebook": notebook,
        "base": {**base, "parameter_count": log["parameter_count"]},
        "output": output,
        "environment": environment,
        "configuration": {
            "epochs": log["epochs"],
            "micro_batch": s["micro_batch"],
            "grad_accum": s["grad_accum"],
            "effective_batch": s["micro_batch"] * s["grad_accum"] * environment["gpu_count"],
            "group_size": s["group_size"],
            "lr_encoder": s["lr_encoder"],
            "lr_head": s["lr_head"],
            "weight_decay": s["weight_decay"],
            "schedule": s["schedule"],
            "min_lr": s["min_lr"],
            "grad_clip": s["grad_clip"],
            "seed": s["seed"],
            "loop": {
                "loss": LOSS_TEXT[log["loss_mode"]],
                "cross_entropy_weight": s["cross_entropy_weight"],
                "sigma_start": s["sigma_start"],
                "sigma_end": s["sigma_end"],
                "reward_weight_spherical": s["reward_weight_spherical"],
                "reward_weight_rps": s["reward_weight_rps"],
                "max_len": s["max_len"],
                "head_max_len": s["head_max_len"],
                "max_tokens_per_batch": s["max_tokens_per_batch"],
                "gradient_checkpointing": s["gradient_checkpointing"],
                "mixed_precision": s["mixed_precision"],
                "torch_seed_base": s["torch_seed_base"],
                "holdout_max_items": s["holdout_max_items"],
                "holdout_fraction_percent": s["holdout_fraction_percent"],
            },
            "deviations": deviations,
        },
        "data": {name: manifest[name] for name in DATA_FIELDS},
        "training": {
            "loss_per_epoch": log["loss_per_epoch"],
            "wall_time_seconds": log["wall_time_seconds"],
            "train_fit": log["train_fit"],
            "laya_temperatures": log["laya_temperatures"],
        },
        "leakage_guard": manifest["leakage_guard"],
        "notes": notes,
    }


if __name__ == "__main__":
    import datetime

    import transformers

    # A run that departs from the spec's defaults must say so (run 2 and 3 of the run plan).
    if LOG["loss_mode"] != "full" or LOG["epochs"] != 4:
        assert DEVIATIONS, "set DEVIATIONS: this run's loss or epochs differ from TSD-020 P2"
    notes = (
        f"Rule R (train fit >= {RULE_R_MIN_FIT:.0%} on both questions): "
        f"{'PASS' if RULE_R_PASSED else 'FAIL'}. Option order is fixed (the vendor loop does not "
        "shuffle options). Weights are saved in fp16. The vendor's model_name rename is kept. "
        f"Train items used: {LOG['train_items_used']}; "
        f"temperature hold-out items: {LOG['holdout_items']}."
    )
    RECORD = assemble_run_record(
        run_number=RUN_NUMBER,
        run_date=datetime.date.today().isoformat(),
        notebook=NOTEBOOK_PATH,
        base={
            "repo": BASE_REPO,
            "subfolder": BASE_SUBFOLDER,
            "revision": BASE_REVISION,
            "sha256": BASE_SHA256,
        },
        output={
            "repo": OUTPUT_REPO,
            "revision": OUTPUT_REVISION,
            "tag": TAG,
            "sha256": OUTPUT_SHA256,
        },
        environment={
            "laya_version": laya.__version__,
            "torch_version": torch.__version__,
            "transformers_version": transformers.__version__,
            "cuda_version": torch.version.cuda,
            "gpu_model": torch.cuda.get_device_name(0),
            "gpu_count": torch.cuda.device_count(),
        },
        manifest=MANIFEST,
        log=LOG,
        deviations=DEVIATIONS,
        notes=notes,
    )
    with open(f"{WORK}/run_record.json", "w", encoding="utf-8") as handle:
        json.dump(RECORD, handle, indent=2, sort_keys=True, ensure_ascii=False)
    print("wrote", f"{WORK}/run_record.json")
    print("Download run_record.json and give it to the maintainer: calvino.classifiers")
    print(".finetune_record validates it and write_run_record puts it in reports/finetune/.")